In [ ]:
pip install pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

SUSPICIOUS_LIMIT = 10000.00


In [ ]:
file_url = "../../work/documents/transactions.csv"

try:
    df = pd.read_csv(file_url)

except FileNotFoundError:
    print(f"Erro: arquivo '{file_url}' não encontrado.")
    exit()

In [ ]:
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")

df["data"] = pd.to_datetime(
    df["data"],
    format="%Y-%m-%d",
    errors="coerce"
)

mask_valid = (
    df["id"].notna()
    & df["cliente_id"].notna()
    & (df["cliente_id"].astype(str).str.strip() != "")
    & df["data"].notna()
    & df["tipo"].isin(["credito", "debito"])
    & df["valor"].notna()
    & (df["valor"] > 0)
)

total_read = len(df)

df_valid = df[mask_valid].copy()
df_invalid = df[~mask_valid].copy()

print(f"Total de linhas lidas: {total_read}")
print(f"Linhas válidas: {len(df_valid)}")
print(f"Linhas inválidas: {len(df_invalid)}")

In [ ]:
df_valid["mes"] = df_valid["data"].dt.strftime("%Y-%m")

monthly_report = {}

for month, group in df_valid.groupby("mes"):

    total_credito = group.loc[
        group["tipo"] == "credito",
        "valor"
    ].sum()

    total_debito = group.loc[
        group["tipo"] == "debito",
        "valor"
    ].sum()

    monthly_report[month] = {
        "quantidade": int(len(group)),
        "total_credito": round(float(total_credito), 2),
        "total_debito": round(float(total_debito), 2),
        "saldo": round(float(total_credito - total_debito), 2),
        "media": round(float(group["valor"].mean()), 2),
        "maior_valor": round(float(group["valor"].max()), 2),
        "menor_valor": round(float(group["valor"].min()), 2)
    }

In [ ]:
suspects = df_valid[
    df_valid["valor"] > SUSPICIOUS_LIMIT
]

print("\n===== TRANSAÇÕES SUSPEITAS =====")

if suspects.empty:
    print("Nenhuma transação suspeita encontrada.")
else:
    for _, t in suspects.iterrows():
        valor = (
            f"{t['valor']:,.2f}"
            .replace(",", "X")
            .replace(".", ",")
            .replace("X", ".")
        )

        print(
            f"ID: {int(t['id'])} | "
            f"Cliente: {t['cliente_id']} | "
            f"Data: {t['data'].strftime('%Y-%m-%d')} | "
            f"Valor: R$ {valor}"
        )

In [ ]:
print("\n===== RESUMO MENSAL =====")

for mes, dados in monthly_report.items():
    print(f"\nMês: {mes}")
    for chave, valor in dados.items():
        print(f"{chave}: {valor}")

In [ ]:
months = sorted(monthly_report.keys())
balances = [monthly_report[mes]["saldo"] for mes in months]

plt.figure(figsize=(8, 5))
plt.bar(months, balances)

plt.title("Saldo Mensal das Transações")
plt.xlabel("Mês")
plt.ylabel("Saldo (Crédito - Débito)")

plt.tight_layout()
plt.savefig("grafico.png")
plt.close()

print("Gráfico salvo como grafico.png")

In [ ]:
debitos = [monthly_report[month]["total_debito"] for month in months]

plt.figure(figsize=(8, 5))
plt.plot(months, debitos, marker="o", label="Débitos")

plt.title("Evolução dos Débitos por Mês")
plt.xlabel("Mês")
plt.ylabel("Valor (R$)")
plt.legend()

plt.tight_layout()
plt.savefig("grafico.png")
plt.close()

In [ ]:
months = sorted(monthly_report.keys())

credits = [monthly_report[mes]["total_credito"] for mes in months]
debits = [monthly_report[mes]["total_debito"] for mes in months]

plt.bar(months, credits, label="Crédito")
plt.bar(months, debits, bottom=credits, label="Débito")

plt.title("Créditos e Débitos por Mês")
plt.xlabel("Mês")
plt.ylabel("Valor (R$)")
plt.legend()

plt.tight_layout()
plt.savefig("grafico.png")
plt.close()